# Intelligent Customer Service Agent
## LLM Project 1 — ReAct + LangGraph

This notebook demonstrates all **11 required functions** from Section 9 of the project specification, one cell per test case.

### System Architecture

```
User Input
    |
[Planner Node]   ← ReAct: extract intent + entities, select tool(s)
    |
[Tool Node(s)]   ← MySQL queries / business logic
    |
[Verifier Node]  ← prevent hallucinations, enforce policy
    |
Final Response
```

| Component | Implementation |
|-----------|---------------|
| LLM | OpenAI `gpt-4o-mini` |
| Framework | LangGraph + LangChain |
| Short-Term Memory (STM) | LangGraph `MemorySaver` (per `thread_id`) |
| Long-Term Memory (LTM) | MySQL `customer_memory` table @ `140.118.122.119/llm-course` |
| Tools | 6 tools: order_lookup, customer_profile, request_refund, log_complaint, read/write_long_term_memory |

### Test Score Checklist (Section 9)

| # | Function | Test Query | Expected Behavior |
|---|----------|-----------|-------------------|
| 1 | Intent Parsing | Where is my order 12345? | Extract intent=tracking, order_id=12345 |
| 2 | OrderLookupTool | Check status of order 1001 | MySQL SELECT from orders |
| 3 | CustomerProfileTool | Show my profile | MySQL SELECT from customers |
| 4 | RefundTool | Refund order 5678 | MySQL UPDATE status=refund_requested |
| 5 | ComplaintLoggerTool | Complain about order 2222 | MySQL INSERT into complaints |
| 6 | Multi-step Reasoning | Refund 7890 if delivered | order_lookup → conditional refund |
| 7 | Short-Term Memory (STM) | Cancel it (after prior query) | recall order_id from same thread |
| 8 | Long-Term Memory Read | What issues have I had before? | SELECT customer_memory |
| 9 | Long-Term Memory Write | Remember I prefer refunds | INSERT customer_memory |
| 10 | Personalization | My order is late again | detect repeated issue from LTM |
| 11 | Verifier | Refund order 0000 | reject — order not found |

---
## Setup — Load Agent & Verify Connections

In [ ]:
import sys, os, uuid, warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, os.getcwd())

from main import app, get_db_connection
from langchain_core.messages import HumanMessage, AIMessage

# Verify remote DB connection
try:
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SHOW TABLES")
    tables = [t[0] for t in cursor.fetchall()]
    conn.close()
    print(f"Remote MySQL connected.  Tables: {tables}")
except Exception as e:
    print(f"DB connection failed: {e}")

print("Agent loaded.")
print("STM : LangGraph MemorySaver (per thread_id, in-process)")
print("LTM : MySQL customer_memory  @  140.118.122.119 / llm-course")
print("Tools: order_lookup, customer_profile, request_refund,")
print("       log_complaint, read_long_term_memory, write_long_term_memory")

In [ ]:
def run_query(user_input: str, customer_id: int, thread_id: str = None) -> tuple:
    """Run a query through the ReAct agent and print the full execution trace."""
    if thread_id is None:
        thread_id = f"s_{uuid.uuid4().hex[:6]}"
    cfg = {"configurable": {"thread_id": thread_id, "customer_id": customer_id}}

    sep = "=" * 72
    print(f"\n{sep}")
    print(f"  QUERY   : {user_input}")
    print(f"  Customer: {customer_id}  |  Thread: {thread_id}")
    print(f"{sep}")

    events = app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        cfg,
        stream_mode="values",
    )

    final_response = None
    for event in events:
        msg = event["messages"][-1]
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"\n  [Planner] selected tool : {tc['name']}")
                args_display = {k: v for k, v in tc.get("args", {}).items() if k != "config"}
                print(f"            arguments    : {args_display}")
        elif msg.type == "tool":
            preview = msg.content[:400] + ("..." if len(msg.content) > 400 else "")
            print(f"  [Tool: {msg.name}]")
            print(f"    {preview}")
        elif isinstance(msg, AIMessage) and not msg.tool_calls:
            final_response = msg.content

    print(f"\n  [Final Response — Verifier output]")
    print(f"  {'-' * 60}")
    for line in (final_response or "(no response)").split("\n"):
        print(f"  {line}")
    print()
    return final_response, cfg

print("run_query() helper ready.")

---
## Test 1 — Intent Parsing

**Requirement**: The Planner Node must extract `intent = tracking` and `entity = order_id: 12345` from natural language.

**Query**: `Where is my order 12345?`  
**Customer**: Alice (customer_id=1)  
**Data**: Order 12345 — Wireless Mouse, status: **shipped**  
**Expected**: Planner identifies tracking intent, calls `order_lookup(order_id=12345)`

In [ ]:
run_query("Where is my order 12345?", customer_id=1)

---
## Test 2 — OrderLookupTool

**Requirement**: `order_lookup` tool executes `SELECT * FROM orders WHERE order_id=? AND customer_id=?`

**Query**: `Check status of order 1001`  
**Customer**: Bob (customer_id=2)  
**Data**: Order 1001 — Mechanical Keyboard, status: **processing**  
**Expected**: MySQL SELECT returns order details

In [ ]:
run_query("Check status of order 1001", customer_id=2)

---
## Test 3 — CustomerProfileTool

**Requirement**: `customer_profile` tool executes `SELECT * FROM customers WHERE customer_id=?`

**Query**: `Show my profile`  
**Customer**: Alice (customer_id=1)  
**Expected**: MySQL SELECT returns name, email, created_at

In [ ]:
run_query("Show my profile", customer_id=1)

---
## Test 4 — RefundTool

**Requirement**: `request_refund` tool executes `UPDATE orders SET status='refund_requested' WHERE order_id=? AND customer_id=?`

**Query**: `Refund order 5678`  
**Customer**: Alice (customer_id=1)  
**Data**: Order 5678 — Noise Cancelling Headphones, status: **delivered** → eligible  
**Expected**: MySQL UPDATE sets status to refund_requested

In [ ]:
run_query("Refund order 5678", customer_id=1)

---
## Test 5 — ComplaintLoggerTool

**Requirement**: `log_complaint` tool executes `INSERT INTO complaints (customer_id, order_id, issue, status) VALUES (...)`

**Query**: `I want to complain about order 2222, the item arrived damaged`  
**Customer**: Charlie (customer_id=3)  
**Data**: Order 2222 — Ergonomic Chair, status: **delivered**  
**Expected**: MySQL INSERT into complaints table with status='open'

In [ ]:
run_query("I want to complain about order 2222, the item arrived damaged", customer_id=3)

---
## Test 6 — Multi-step Reasoning

**Requirement**: The Planner must chain tool calls — first verify delivery status, then conditionally refund.

**Query**: `Refund order 7890 only if it has already been delivered`  
**Customer**: Bob (customer_id=2)  
**Data**: Order 7890 — USB-C Hub, status: **delivered**  
**Expected**: Planner calls `order_lookup` → sees `delivered` → calls `request_refund`

> This demonstrates the ReAct loop: Reason (check delivery) → Act (refund).

In [ ]:
run_query("Refund order 7890 only if it has already been delivered", customer_id=2)

---
## Test 7 — Short-Term Memory (STM)

**Requirement**: The same `thread_id` session retains conversation history via LangGraph `MemorySaver`. The agent must resolve pronoun references across turns.

**Turn 1**: `What is the status of order 1001?` — establishes order context in STM  
**Turn 2**: `Cancel it` — agent resolves `"it"` → order 1001 from STM

**Customer**: Bob (customer_id=2)  
**Expected**: Turn 2 calls `request_refund(order_id=1001)` without being told the order number again

> Both calls use the **same `thread_id`** to share session memory.

In [ ]:
STM_THREAD = f"stm_demo_{uuid.uuid4().hex[:6]}"

print(">>> TURN 1: Establish order context in STM")
run_query("What is the status of order 1001?", customer_id=2, thread_id=STM_THREAD)

In [ ]:
print(f">>> TURN 2: Agent resolves 'it' from STM (same thread: {STM_THREAD})")
run_query("Cancel it", customer_id=2, thread_id=STM_THREAD)

---
## Test 8 — Long-Term Memory (Read)

**Requirement**: `read_long_term_memory` tool queries `SELECT key, value FROM customer_memory WHERE customer_id=?` from the persistent MySQL LTM store at `140.118.122.119`.

**Query**: `What issues have I had before?`  
**Customer**: Charlie (customer_id=3)  
**Pre-seeded LTM**: `past_issues: frequent late deliveries`  
**Expected**: Agent retrieves and reports the stored memory from the remote DB

In [ ]:
run_query("What issues have I had before?", customer_id=3)

---
## Test 9 — Long-Term Memory (Write)

**Requirement**: `write_long_term_memory` tool executes `INSERT INTO customer_memory (customer_id, key, value)` to persist preferences across sessions in the remote MySQL DB.

**Query**: `Remember I prefer refunds over store credit`  
**Customer**: Alice (customer_id=1)  
**Expected**: MySQL INSERT into `customer_memory` table — verifiable by querying the DB directly

In [ ]:
run_query("Remember I prefer refunds over store credit", customer_id=1)

In [ ]:
# Verify the write in the remote DB
conn = get_db_connection()
cursor = conn.cursor(dictionary=True)
cursor.execute("SELECT * FROM customer_memory WHERE customer_id = 1 ORDER BY created_at DESC")
rows = cursor.fetchall()
conn.close()
print("customer_memory rows for Alice (customer_id=1):")
for r in rows:
    print(f"  [{r['id']}] key={r['key']!r:40s} value={r['value']!r}  (at {r['created_at']})")

---
## Test 10 — Personalization

**Requirement**: The agent reads LTM to detect a repeated issue pattern and personalizes its response accordingly.

**Query**: `My order is late again!`  
**Customer**: Charlie (customer_id=3)  
**LTM**: `past_issues: frequent late deliveries`  
**Expected**: Agent calls `read_long_term_memory`, detects the repeated pattern, responds with elevated empathy and priority acknowledgment

In [ ]:
run_query("My order is late again!", customer_id=3)

---
## Test 11 — Verifier Node

**Requirement**: The Verifier Node must prevent hallucinated responses. When `order_lookup` returns "not found", the agent must reject the refund request — not invent a success message.

**Query**: `Refund order 0000`  
**Customer**: Alice (customer_id=1)  
**Data**: Order 0000 does **not exist** in the database  
**Expected**: Tool returns "not found" → Verifier rewrites response to safely reject the request

In [ ]:
run_query("Refund order 0000", customer_id=1)

---
## Summary — All 11 Functions Demonstrated

| # | Function | Tool / Mechanism | MySQL Operation | DB |
|---|----------|-----------------|----------------|----|
| 1 | Intent Parsing | Planner (LLM reasoning) | SELECT orders | remote |
| 2 | OrderLookupTool | `order_lookup` | SELECT orders | remote |
| 3 | CustomerProfileTool | `customer_profile` | SELECT customers | remote |
| 4 | RefundTool | `request_refund` | UPDATE orders | remote |
| 5 | ComplaintLoggerTool | `log_complaint` | INSERT complaints | remote |
| 6 | Multi-step Reasoning | Planner → tool chaining | SELECT + UPDATE | remote |
| 7 | Short-Term Memory | LangGraph MemorySaver | — (in-process) | in-memory |
| 8 | LTM Read | `read_long_term_memory` | SELECT customer_memory | remote |
| 9 | LTM Write | `write_long_term_memory` | INSERT customer_memory | remote |
| 10 | Personalization | LTM read + Planner reasoning | SELECT customer_memory | remote |
| 11 | Verifier | Verifier Node (LLM) | SELECT orders (not found) | remote |

**Database**: All data (customers, orders, complaints, long-term memory) stored on remote MySQL @ `140.118.122.119/llm-course`

> **Demo tip**: Re-run any cell to repeat that test live. Test 7 requires running Turn 1 before Turn 2 to populate STM.